In [ ]:
import json,pickle,zipfile,re,math,os,time
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
    
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
 
    return bpe_data, encode
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# MODEL V2

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RoPE(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V5

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  

        router_logits = self.router(x_flat)                     
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)          
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1)  
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          
        output = (expert_outputs * gate).sum(dim=1)                                        
        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)      
        mean_router_prob = router_probs.mean(dim=0)              
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present

class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params

# MODEL V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    return (n_layer - 1 - layer_idx) % global_every_n == 0


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size   

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                              
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)               
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x

class DecoderOnlyLM_HybridMask(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V7

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,         
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12, 
    n_kv_head=3,
    pad_id=PAD_ID,                 
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"      
LAYERS_PATH = "blocks"           
FINAL_NORM  = ("ln_f",)      
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "statute_id")
 
def _as_label_list(v):
    if v is None:
        return None
    if isinstance(v, (int, np.integer)):
        return [int(v)]
    if isinstance(v, str):
        v = v.replace(",", " ").split()
    return [int(x) for x in v]
 
 
def normalize_docs(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            raw = raw["data"]
        else:
            raw = [{"id": k, **(v if isinstance(v, dict) else {"text": v})} for k, v in raw.items()]
    out = []
    for d in raw:
        text = d.get("text")
        if text is None:
            raise ValueError(f"document {d.get('id')!r} has no 'text' field; keys: {list(d)}")
        if isinstance(text, list):
            text = " ".join(str(x) for x in text)
        lab = _as_label_list(next((d[k] for k in LABEL_KEYS if k in d), None))
        out.append({"id": str(d["id"]), "text": text, "labels": sorted(set(lab)) if lab is not None else None})
    return out
 
 
def lsi_paths(root, include_test=False):
    p = {"train": f"{root}/Train/train.json", "dev": f"{root}/Train/dev.json"}
    if include_test:
        p["test"] = f"{root}/Test/test.json"
    return p
 
 
def download_lsi_files(repo="Exploration-Lab/CS-781-Capstone", p5="Phase_5_LSI", token=None, include_test=False):
    return {k: hf_hub_download(repo_id=repo, filename=f, repo_type="dataset", token=token)for k, f in lsi_paths(p5, include_test).items()}
 
 
def tokenize_docs(docs, encode_fn, clean_fn=None, max_doc_tokens=None, desc="tokenizing"):
    out = []
    for c in tqdm(docs, desc=desc, leave=False):
        t = clean_fn(c["text"]) if clean_fn else c["text"]
        ids = np.asarray(encode_fn(t), dtype=np.int32)
        if max_doc_tokens and len(ids) > max_doc_tokens:
            ids = ids[-max_doc_tokens:]
        out.append({"id": c["id"], "labels": c["labels"], "ids": ids, "n_chars": len(c["text"])})
    return out
 
 
def label_counts(docs, num_labels):
    return np.bincount(np.concatenate([d["y"] for d in docs]) if docs else np.zeros(0, dtype=np.int64),minlength=num_labels).astype(np.int64)
 
 
def docs_to_Y(docs, num_labels):
    Y = np.zeros((len(docs), num_labels), dtype=np.uint8)
    for i, d in enumerate(docs):
        Y[i, d["y"]] = 1
    return Y
 
 
def setup_data(paths, tokenizer_pkl, sep_token="</s>", clean=True, clean_fn=None, max_doc_tokens=None,cache=None, include_test=False):
    if isinstance(paths, str):
        paths = lsi_paths(paths, include_test)
    bpe_data, encode = load_tokenizer(tokenizer_pkl)
    stoi = bpe_data["stoi"]
    pad_id = stoi["<pad>"]
    print("special tokens:", [t for t in stoi if isinstance(t, str)])
    sep_id = stoi.get(sep_token) if sep_token else None
    if sep_token and sep_id is None:
        raise ValueError(f"{sep_token!r} not in the vocabulary; pass sep_token=<one of the special tokens above>")
    clean_fn = (clean_fn or clean_text) if clean else None
    splits = ["train", "dev"] + (["test"] if include_test and "test" in paths else [])
    key = {"max_doc_tokens": max_doc_tokens, "clean": bool(clean_fn), "splits": splits}
    D = {"pad_id": pad_id, "sep_id": sep_id, "vocab_size": bpe_data["vocab_size"], "encode": encode,
         "clean": clean_fn, "max_doc_tokens": max_doc_tokens, "bpe_data": bpe_data}
    cached = None
    if cache and os.path.exists(cache):
        cached = pickle.load(open(cache, "rb"))
        if cached.get("key") != key:
            print("cache settings differ -> re-tokenizing")
            cached = None
        else:
            print(f"loaded tokenized documents from {cache}")
    for split in splits:
        if cached:
            D[split] = cached[split]
        else:
            docs = normalize_docs(json.load(open(paths[split], encoding="utf-8")))
            D[split] = tokenize_docs(docs, encode, clean_fn, max_doc_tokens, desc=f"tokenizing {split}")
    if cache and not cached:
        pickle.dump({"key": key, **{s: D[s] for s in splits}}, open(cache, "wb"))
        print(f"saved tokenized documents to {cache}")
 
    for s in ("train", "dev"):
        assert all(d["labels"] is not None for d in D[s]), f"some {s} documents have no labels field (keys tried: {LABEL_KEYS})"
    labels = sorted({l for s in ("train", "dev") for d in D[s] for l in d["labels"]})
    D["labels"], D["label2idx"], D["num_labels"] = labels, {l: i for i, l in enumerate(labels)}, len(labels)
    for s in splits:
        for d in D[s]:
            d["y"] = np.array([D["label2idx"][l] for l in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
    D["pos_counts"] = label_counts(D["train"], D["num_labels"])
 
    for split in splits:
        docs = D[split]
        lens = np.array([len(d["ids"]) for d in docs])
        line = f"{split:5s}: {len(docs)} docs, tokens/doc median {int(np.median(lens))} p90 {int(np.percentile(lens, 90))} max {lens.max()}, " \
               f"docs over 1024 tokens: {100 * np.mean(lens > 1024):.0f}%"
        if split != "test":
            k = np.array([len(d["labels"]) for d in docs])
            line += f" | statutes/doc mean {k.mean():.2f} median {int(np.median(k))} max {k.max()}, single-statute docs {100 * np.mean(k == 1):.0f}%"
        print(line)
    pc = D["pos_counts"]
    dev_only = sorted(set(l for d in D["dev"] for l in d["labels"]) - set(l for d in D["train"] for l in d["labels"]))
    print(f"label space: {D['num_labels']} statutes (train+dev). Training positives per statute: min {pc.min()}, "
          f"median {int(np.median(pc))}, max {pc.max()}; statutes with <=5 training positives: {int((pc <= 5).sum())}; "
          f"statutes seen only in dev : {len(dev_only)}")
    print(f"PAD_ID={pad_id}  VOCAB_SIZE={D['vocab_size']}  marker token id={sep_id}  cleaning={'on' if clean_fn else 'off'}")
    return D

In [ ]:
DOC_TAIL = dict(mode="tail", max_len=1024)
DOC_HEAD = dict(mode="head", max_len=1024)
DOC_HEADTAIL = dict(mode="headtail", max_len=1024, head_tokens=256)
DOC_CHUNKS = dict(mode="chunks", max_len=1024, stride=None, max_chunks=4, select="even", train_select="random")
 
 
def chunk_windows(n, size, stride):
    stride = max(1, min(stride, size))
    if n <= size:
        return [(0, n)]
    wins, s = [], n - size
    while s > 0:
        wins.append((s, s + size))
        s -= stride
    wins.append((0, size))
    return wins[::-1]
 
 
def select_windows(wins, max_chunks, select, rng=None):
    if not max_chunks or len(wins) <= max_chunks:
        return wins
    if select == "last":
        return wins[-max_chunks:]
    if select == "first":
        return wins[:max_chunks]
    if select == "even":
        idx = sorted(set(np.linspace(0, len(wins) - 1, max_chunks).round().astype(int).tolist()))
        return [wins[i] for i in idx]
    if select == "random":
        rng = rng or np.random.default_rng()
        idx = sorted(rng.choice(len(wins), size=max_chunks, replace=False).tolist())
        return [wins[i] for i in idx]
    raise ValueError(f"unknown select {select!r}")
 
 
def doc_to_chunks(ids, DOC, sep_id, training=False, rng=None):
    size = DOC["max_len"] - 1
    n, mode = len(ids), DOC["mode"]
    if mode == "tail":
        parts = [ids[-size:]]
    elif mode == "head":
        parts = [ids[:size]]
    elif mode == "headtail":
        if n <= size:
            parts = [ids]
        else:
            h = DOC.get("head_tokens", size // 4)
            parts = [np.concatenate([ids[:h], ids[-(size - h):]])]
    elif mode == "chunks":
        wins = chunk_windows(n, size, DOC.get("stride") or size)
        sel = DOC.get("train_select", DOC["select"]) if training else DOC["select"]
        parts = [ids[a:b] for a, b in select_windows(wins, DOC.get("max_chunks"), sel, rng)]
    else:
        raise ValueError(f"unknown mode {mode!r}")
    return [np.append(p, sep_id).astype(np.int64) for p in parts]
 
 
def collate_lsi(batch, pad_id, DOC, sep_id, num_labels, training=False, rng=None):
    chunks, doc_idx, chunk_pos, lengths = [], [], [], []
    for b_i, item in enumerate(batch):
        for k, c in enumerate(doc_to_chunks(item["ids"], DOC, sep_id, training, rng)):
            chunks.append(c); doc_idx.append(b_i); chunk_pos.append(k); lengths.append(len(c))
    ids = torch.full((len(chunks), max(lengths)), pad_id, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c)
    Y = torch.zeros(len(batch), num_labels)
    for i, it in enumerate(batch):
        if it.get("y") is not None:
            Y[i, torch.from_numpy(it["y"])] = 1.0
    return {"input_ids": ids, "lengths": torch.tensor(lengths), "doc_idx": torch.tensor(doc_idx),"chunk_pos": torch.tensor(chunk_pos), "n_docs": len(batch), "labels": Y}

In [ ]:
def load_phase1(build_fn, path):
    model = build_fn()
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return model
 
 
class LSIModel(nn.Module):
    def __init__(self, backbone, num_labels, head_path="lm_head", pool="last", agg="mean", dropout=0.1, prior=None):
        super().__init__()
        self.backbone = backbone
        parent = backbone
        *parents, name = head_path.split(".")
        for p in parents:
            parent = getattr(parent, p)
        setattr(parent, name, nn.Identity())
        emb = next(m for m in backbone.modules() if isinstance(m, nn.Embedding))
        hidden = emb.embedding_dim
        self.vocab_size, self.num_labels, self.pool, self.agg = emb.num_embeddings, num_labels, pool, agg
        self.drop = nn.Dropout(dropout)
        self.att = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1)) if agg == "attn" else None
        self.classifier = nn.Linear(hidden, num_labels)
        if prior is not None:
            p = np.clip(np.asarray(prior, dtype=np.float64), 1e-4, 1 - 1e-4)
            with torch.no_grad():
                self.classifier.bias.copy_(torch.tensor(np.log(p / (1 - p)), dtype=torch.float))
 
    def head_parameters(self):
        ps = list(self.classifier.parameters())
        if self.att is not None:
            ps += list(self.att.parameters())
        return ps
 
    def encode_chunks(self, input_ids, lengths):
        out = self.backbone(input_ids)
        h = out["logits"] if isinstance(out, dict) else out                      
        if self.pool == "last":
            idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, h.size(-1))
            return h.gather(1, idx).squeeze(1)
        m = (torch.arange(h.size(1), device=h.device)[None, :] < lengths[:, None]).unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)
 
    def forward(self, input_ids, lengths, doc_idx, chunk_pos, n_docs):
        v = self.encode_chunks(input_ids, lengths)                                
        Cmax = int(chunk_pos.max().item()) + 1
        mask = torch.zeros(n_docs, Cmax, dtype=torch.bool, device=v.device).index_put(
            (doc_idx, chunk_pos), torch.ones_like(doc_idx, dtype=torch.bool))
        if self.agg in ("logit_max", "logit_mean"):
            lg = self.classifier(self.drop(v))                                    
            base = lg.new_zeros(n_docs, Cmax, lg.size(-1)).index_put((doc_idx, chunk_pos), lg)
            if self.agg == "logit_max":
                return base.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(1).values
            mf = mask.unsqueeze(-1).to(base.dtype)
            return (base * mf).sum(1) / mf.sum(1).clamp(min=1)
        pad = v.new_zeros(n_docs, Cmax, v.size(-1)).index_put((doc_idx, chunk_pos), v)
        mf = mask.unsqueeze(-1).to(pad.dtype)
        if self.agg == "mean":
            d = (pad * mf).sum(1) / mf.sum(1).clamp(min=1)
        elif self.agg == "max":
            d = pad.masked_fill(~mask.unsqueeze(-1), float("-inf")).max(1).values
        elif self.agg == "attn":
            s = self.att(pad).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            d = (torch.softmax(s, dim=1).unsqueeze(-1).to(pad.dtype) * pad).sum(1)
        else:                                                                    
            d = pad[torch.arange(n_docs, device=pad.device), mask.sum(1) - 1]
        return self.classifier(self.drop(d))                                      
 
 
def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
 
 
def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot
 
 
def dummy_lsi_batch(vocab_size, device, T=32):
    ids = torch.randint(0, vocab_size, (2, T), device=device)
    return (ids, torch.tensor([T, T - 12], device=device), torch.tensor([0, 1], device=device),torch.tensor([0, 0], device=device), 2)

In [ ]:
def make_pos_weight(pos_counts, n_docs, mode, cap=30.0):
    if not mode:
        return None
    pos = np.maximum(np.asarray(pos_counts, dtype=np.float64), 1.0)
    w = (n_docs - np.asarray(pos_counts, dtype=np.float64)).clip(min=1.0) / pos
    if mode == "sqrt":
        w = np.sqrt(w)
    return torch.tensor(np.clip(w, 1.0, cap), dtype=torch.float)
 
 
def lsi_loss(logits, Y, loss="bce", pos_weight=None, gamma=2.0):
    logits = logits.float()
    if loss == "bce":
        return F.binary_cross_entropy_with_logits(logits, Y, pos_weight=pos_weight)
    if loss == "focal":
        bce = F.binary_cross_entropy_with_logits(logits, Y, pos_weight=pos_weight, reduction="none")
        p = torch.sigmoid(logits)
        pt = Y * p + (1 - Y) * (1 - p)
        return ((1 - pt) ** gamma * bce).mean()
    raise ValueError(f"unknown loss {loss!r}")
 

In [ ]:
def multilabel_scores(Y, pred):
    Y, P = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tp, fp, fn = (Y & P).sum(0), (~Y & P).sum(0), (Y & ~P).sum(0)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    prec, rec = tp / np.maximum(tp + fp, 1), tp / np.maximum(tp + fn, 1)
    support = Y.sum(0)
    present = support > 0
    mean = lambda a: float(a[present].mean()) if present.any() else 0.0
    return {"macro_f1": mean(f1), "macro_precision": mean(prec), "macro_recall": mean(rec),
            "macro_f1_all": float(f1.mean()),
            "micro_f1": float(2 * tp.sum() / max(1, 2 * tp.sum() + fp.sum() + fn.sum())),
            "subset_acc": float((Y == P).all(1).mean()),
            "per_label": {"f1": f1, "precision": prec, "recall": rec, "support": support},
            "avg_pred_per_doc": float(P.sum(1).mean()), "avg_gold_per_doc": float(Y.sum(1).mean())}
 
 
def decode_probs(probs, thresholds, min_labels=1, max_labels=None):
    probs = np.asarray(probs, dtype=np.float32)
    thr = np.broadcast_to(np.asarray(thresholds, dtype=np.float32), (probs.shape[1],))
    pred = probs >= thr[None, :]
    ratio = probs / np.maximum(thr[None, :], 1e-6)
    if min_labels:
        for i in np.where(pred.sum(1) < min_labels)[0]:
            pred[i, np.argsort(-ratio[i])[:min_labels]] = True
    if max_labels:
        for i in np.where(pred.sum(1) > max_labels)[0]:
            keep = np.argsort(-np.where(pred[i], ratio[i], -np.inf))[:max_labels]
            pred[i] = False
            pred[i, keep] = True
    return pred
 
 
DEFAULT_GRID = np.round(np.linspace(0.02, 0.90, 45), 3)
 
def tune_global_threshold(probs, Y, grid=None, metric="macro_f1", min_labels=1):
    best_t, best_s, best = None, -1.0, None
    for t in (DEFAULT_GRID if grid is None else grid):
        s = multilabel_scores(Y, decode_probs(probs, t, min_labels))
        if s[metric] > best_s:
            best_t, best_s, best = float(t), s[metric], s
    return best_t, best
 
 
def tune_label_thresholds(probs, Y, global_t, grid=None, min_pos=5):
    grid = DEFAULT_GRID if grid is None else grid
    Yb = np.asarray(Y).astype(bool)
    thr = np.full(Yb.shape[1], global_t, dtype=np.float32)
    for l in range(Yb.shape[1]):
        npos = int(Yb[:, l].sum())
        if npos < min_pos:
            continue
        best_f, best_t = -1.0, global_t
        for t in grid:
            p = probs[:, l] >= t
            tp = int((p & Yb[:, l]).sum())
            den = 2 * tp + int(p.sum()) - tp + npos - tp
            f = 2 * tp / den if den else 0.0
            if f > best_f + 1e-12 or (abs(f - best_f) <= 1e-12 and abs(t - global_t) < abs(best_t - global_t)):
                best_f, best_t = f, float(t)
        thr[l] = best_t
    return thr
 
 
def print_scores(s, name=""):
    print(f"{name} macro-F1 {s['macro_f1']:.4f} (P {s['macro_precision']:.4f}, R {s['macro_recall']:.4f}) | "
          f"macro-F1 over all statutes {s['macro_f1_all']:.4f} | micro-F1 {s['micro_f1']:.4f} | subset acc {s['subset_acc']:.4f} | "
          f"predicted {s['avg_pred_per_doc']:.2f} vs gold {s['avg_gold_per_doc']:.2f} statutes/doc")
 
 
def bucket_report(s, pos_counts, edges=(0, 5, 20, 100, 10 ** 9)):
    f1, sup = s["per_label"]["f1"], s["per_label"]["support"]
    pc = np.asarray(pos_counts)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pc > lo) & (pc <= hi) & (sup > 0) if lo > 0 else (pc <= hi) & (sup > 0)
        name = f"<= {hi}" if lo == 0 else (f"> {lo}" if hi >= 10 ** 9 else f"{lo + 1}-{hi}")
        print(f"  train positives {name:>8s}: {int(m.sum()):4d} statutes, mean F1 {f1[m].mean() if m.any() else float('nan'):.3f}")


In [ ]:
@torch.no_grad()
def predict_lsi(model, docs, DOC, pad_id, sep_id, device, batch_size=2, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    L = model.num_labels
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["ids"]))
    probs = np.zeros((len(docs), L), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idxs = order[s:s + batch_size]
        b = collate_lsi([docs[i] for i in idxs], pad_id, DOC, sep_id, L, training=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(b["input_ids"].to(device), b["lengths"].to(device), b["doc_idx"].to(device),
                           b["chunk_pos"].to(device), b["n_docs"])
        probs[idxs] = torch.sigmoid(logits.float()).cpu().numpy()
    return probs
 
 
CFG_LSI = dict(epochs=5, batch_size=1, grad_accum=16, lr_backbone=2e-5, lr_head=1e-3, weight_decay=0.01,warmup_ratio=0.1, max_grad_norm=1.0, fp16=True, seed=0,loss="bce", pos_weight=None,drop_out=0.1 focal_gamma=2.0)    
 
 
def train_lsi(model, train_docs, pad_id, sep_id, DOC, cfg, device, val_fn=None, save_path=None, pos_weight=None,extra_loss=None, on_step=None):
    model.to(device)
    rng = np.random.default_rng(cfg.get("seed", 0))
    L = model.num_labels
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
 
    loader = DataLoader(train_docs, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_lsi(b, pad_id, DOC, sep_id, L, training=True, rng=rng))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)
 
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
 
    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    pw = pos_weight.to(device) if pos_weight is not None else None
    best, history, step = -1.0, [], 0
    for epoch in range(cfg["epochs"]):
        model.train()
        run, nb, t0 = 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            ids, ln, di, cp = (b[k].to(device) for k in ("input_ids", "lengths", "doc_idx", "chunk_pos"))
            Y = b["labels"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(ids, ln, di, cp, b["n_docs"])
            loss = lsi_loss(logits, Y, cfg.get("loss", "bce"), pw, cfg.get("focal_gamma", 2.0))
            if extra_loss is not None:
                loss = loss + extra_loss(model)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step()
                step += 1
                if on_step is not None:
                    on_step(model, step)
                opt.zero_grad()
        rec = {"epoch": epoch + 1, "train_loss": run / max(1, nb), "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history
 
 
def lsi_report(model, D, DOC, device, batch_size=2, min_pos=5):
    probs = predict_lsi(model, D["dev"], DOC, D["pad_id"], D["sep_id"], device, batch_size)
    Y = docs_to_Y(D["dev"], D["num_labels"])
    t, s_glob = tune_global_threshold(probs, Y, min_labels=1)
    print(f"global threshold {t:.2f}"); print_scores(s_glob, "  global thr, >=1 statute/doc:")
    print_scores(multilabel_scores(Y, decode_probs(probs, t, min_labels=0)), "  global thr, no fallback    :")
    thr = tune_label_thresholds(probs, Y, t, min_pos=min_pos)
    s_lab = multilabel_scores(Y, decode_probs(probs, thr, min_labels=1))
    print_scores(s_lab, f"  per-statute thr (>= {min_pos} dev positives; optimistic, tuned on dev):")
    bucket_report(s_glob, D["pos_counts"])
    return {"global_t": t, "label_thr": thr, "scores_global": s_glob, "scores_label": s_lab, "probs": probs}
 
 
def predict_cases(model, raw_cases, D, DOC, device, thresholds, min_labels=1, batch_size=2):
    cases = normalize_docs(raw_cases)
    docs = tokenize_docs(cases, D["encode"], D["clean"], D["max_doc_tokens"], desc="tokenizing cases")
    probs = predict_lsi(model, docs, DOC, D["pad_id"], D["sep_id"], device, batch_size=batch_size)
    pred = decode_probs(probs, thresholds, min_labels)
    ids = D["labels"]
    out = [{"id": c["id"], "pred": sorted({int(ids[j]) for j in np.where(pred[i])[0]})} for i, c in enumerate(cases)]
    return out
 
 
def write_lsi_submission(preds, valid_ids=None, out_dir="/kaggle/working", zip_name="submission_lsi.zip"):
    valid = set(valid_ids) if valid_ids is not None else None
    path = os.path.join(out_dir, "prediction.json")
    json.dump(preds, open(path, "w", encoding="utf-8"))
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {zp}: {len(preds)} documents, {np.mean([len(p['pred']) for p in preds]):.2f} statutes/document on average")
 

In [ ]:
def run_full_ft_lsi(build_fn, model_pt, head_path, D, DOC, device, tag="lsi_full", pool="last", agg="mean",cfg=None, train_subset=None, eval_batch_size=2):
    cfg = cfg or CFG_LSI
    L = D["num_labels"]
    train_docs = D["train"]
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(train_docs), size=min(train_subset, len(train_docs)), replace=False)
        train_docs = [train_docs[i] for i in sorted(pick)]
    counts = label_counts(train_docs, L)
    prior = (counts + 0.5) / (len(train_docs) + 1)                    
    model = LSIModel(load_phase1(build_fn, model_pt), L, head_path=head_path, pool=pool, agg=agg, prior=prior)
    for p in model.parameters():
        p.requires_grad = True
    count_params(model)
    print(f"training on {len(train_docs)} documents, {L} statutes; document representation: {DOC}; pool={pool}, agg={agg}"f"loss={cfg.get('loss', 'bce')}, pos_weight={cfg.get('pos_weight')}")

    Y_dev = docs_to_Y(D["dev"], L)

    def val_fn(m):
        probs = predict_lsi(m, D["dev"], DOC, D["pad_id"], D["sep_id"], device, eval_batch_size)
        t, s = tune_global_threshold(probs, Y_dev, min_labels=1)
        print_scores(s, f"   dev (thr {t:.2f}):")
        return s["macro_f1"]

    pw = make_pos_weight(counts, len(train_docs), cfg.get("pos_weight"))
    hist = train_lsi(model, train_docs, D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", pos_weight=pw)
    return model, hist

In [ ]:
def adalora_budget(step, total_step, tinit, tfinal, init_bgt, target_bgt, deltaT):
    if step <= tinit:
        return init_bgt, False
    if step > total_step - tfinal:
        return target_bgt, True
    mul = 1.0 - (step - tinit) / (total_step - tfinal - tinit)
    return int((init_bgt - target_bgt) * mul ** 3 + target_bgt), (step % deltaT == 0)


def ema_update(avg, unc, ipt, beta1, beta2):
    avg = beta1 * avg + (1.0 - beta1) * ipt
    unc = beta2 * unc + (1.0 - beta2) * np.abs(ipt - avg)
    return avg, unc


def triplet_scores(sA, sE, sB):
    return sE + sA.mean(axis=1) + sB.mean(axis=0)


def select_prune(scores_per_layer, n_prune):
    flat = np.concatenate(scores_per_layer)
    keep = np.ones(len(flat), dtype=bool)
    if n_prune > 0:
        keep[np.argsort(flat, kind="stable")[:n_prune]] = False
    out, i = [], 0
    for s in scores_per_layer:
        out.append(keep[i:i + len(s)])
        i += len(s)
    return out


class AdaLoRAPlanner:
    def __init__(self, n_layers, init_r, target_r, total_step, tinit=0, tfinal=0, deltaT=1, beta1=0.85, beta2=0.85):
        assert 0 < target_r <= init_r, "need 0 < target_r <= init_r"
        assert tinit + tfinal < total_step, "tinit + tfinal must be < total_step (there must be a budgeting phase)"
        self.n, self.init_r = n_layers, init_r
        self.init_bgt, self.target_bgt = n_layers * init_r, n_layers * target_r
        self.total_step, self.tinit, self.tfinal, self.deltaT = total_step, tinit, tfinal, max(1, deltaT)
        self.beta1, self.beta2 = beta1, beta2
        self.avg, self.unc, self.final_keep = {}, {}, None

    def update_importance(self, params, grads):
        for l, (p, g) in enumerate(zip(params, grads)):
            for k in ("A", "E", "B"):
                ipt = np.abs(p[k] * g[k])
                a, u = self.avg.get((l, k), np.zeros_like(ipt)), self.unc.get((l, k), np.zeros_like(ipt))
                self.avg[(l, k)], self.unc[(l, k)] = ema_update(a, u, ipt, self.beta1, self.beta2)

    def scores(self):
        s = lambda l, k: self.avg[(l, k)] * self.unc[(l, k)]
        return [triplet_scores(s(l, "A"), s(l, "E"), s(l, "B")) for l in range(self.n)]

    def plan(self, step):
        if step > self.total_step - self.tfinal:                 
            return self.final_keep
        if not self.avg:                                         
            return None
        budget, mask_now = adalora_budget(step, self.total_step, self.tinit, self.tfinal,
                                          self.init_bgt, self.target_bgt, self.deltaT)
        if step == self.total_step - self.tfinal:               
            self.final_keep = select_prune(self.scores(), self.init_bgt - budget)
            return self.final_keep
        return select_prune(self.scores(), self.init_bgt - budget) if mask_now else None


def lsi_total_steps(train_docs, cfg):
    n_batches = math.ceil(len(train_docs) / cfg["batch_size"])
    return max(1, (n_batches // cfg["grad_accum"]) * cfg["epochs"])


def _wrap_linears(root, targets, make):
    names = [n for n, m in root.named_modules()
             if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    assert names, f"no nn.Linear matched {targets}"
    for n in names:
        parent_path, _, child = n.rpartition(".")
        parent = root.get_submodule(parent_path) if parent_path else root
        setattr(parent, child, make(getattr(parent, child)))
    return names


_SHIFT_IDX = {}


def _shift_index(d, device):
    key = (d, str(device))
    if key not in _SHIFT_IDX:
        i = torch.arange(d, device=device)
        _SHIFT_IDX[key] = (i[None, :] - i[:, None]) % d          
    return _SHIFT_IDX[key]


class LoRMALinear(nn.Module):
    def __init__(self, base, variant="plus", r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert variant in ("plus", "pi") and isinstance(base, nn.Linear)
        self.base, self.variant, self.r = base, variant, r
        d = base.out_features
        self.scale = alpha / r if variant == "plus" else 1.0     
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False
        self.reset_parameters()

    @torch.no_grad()
    def reset_parameters(self):
        if self.variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B.zero_()
        else:
            self.A.zero_(); self.A[0, 0] = 1.0
            self.B.zero_(); self.B[:, 0] = 1.0
            if self.r > 1:
                nn.init.kaiming_uniform_(self.B[:, 1:], a=math.sqrt(5))

    def _pi_matrix(self):
        BA = self.B @ self.A
        return torch.gather(BA, 1, _shift_index(BA.size(0), BA.device))

    def forward(self, x):
        y = F.linear(x, self.base.weight)                        
        if self.variant == "plus":
            y = y + self.scale * F.linear(F.linear(self.drop(y), self.A), self.B)
        else:
            y = y @ self._pi_matrix().t().to(y.dtype)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):                                      
        if self.variant == "plus":
            d = self.A.size(1)
            M = torch.eye(d, device=self.A.device, dtype=self.A.dtype) + self.scale * (self.B @ self.A)
        else:
            M = self._pi_matrix()
        return M @ self.base.weight

    @torch.no_grad()
    def to_linear(self):
        W = self.merged_weight()
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin


def inject_lorma(root, targets=("q_proj", "v_proj"), variant="plus", r=8, alpha=16, dropout=0.0):
    names = _wrap_linears(root, targets, lambda base: LoRMALinear(base, variant, r, alpha, dropout))
    print(f"LoRMA-{variant}: wrapped {len(names)} layers, e.g. {names[:3]}")
    return names


class SVDLinear(nn.Module):
    def __init__(self, base, init_r=12, alpha=16, dropout=0.0, init_std=0.02):
        super().__init__()
        assert isinstance(base, nn.Linear)
        self.base, self.init_r, self.scale = base, init_r, alpha / init_r
        self.A = nn.Parameter(torch.randn(init_r, base.in_features) * init_std)
        self.E = nn.Parameter(torch.zeros(init_r))
        self.B = nn.Parameter(torch.randn(base.out_features, init_r) * init_std)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        z = F.linear(self.drop(x), self.A)                       
        z = z * self.E.to(z.dtype)                                
        return self.base(x) + self.scale * F.linear(z, self.B)

    @torch.no_grad()
    def to_linear(self):
        W = self.base.weight + self.scale * (self.B * self.E[None, :]) @ self.A
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin


def inject_adalora(root, targets=("q_proj", "v_proj"), init_r=12, alpha=16, dropout=0.0, init_std=0.02):
    names = _wrap_linears(root, targets, lambda base: SVDLinear(base, init_r, alpha, dropout, init_std))
    print(f"AdaLoRA: wrapped {len(names)} layers at init_r={init_r}, e.g. {names[:3]}")
    return names


def adalora_orth_reg(model):
    regs = []
    for m in model.modules():
        if isinstance(m, SVDLinear):
            I = torch.eye(m.init_r, device=m.A.device, dtype=m.A.dtype)
            regs.append(((m.A @ m.A.T - I) ** 2).sum() + ((m.B.T @ m.B - I) ** 2).sum())
    return torch.stack(regs).mean() if regs else torch.zeros(())


class AdaLoRAAllocator:
    def __init__(self, model, total_step, target_r, tinit=0, tfinal=0, deltaT=1, beta1=0.85, beta2=0.85):
        self.mods = [m for m in model.modules() if isinstance(m, SVDLinear)]
        assert self.mods, "no SVDLinear layers found: call inject_adalora first"
        assert len({m.init_r for m in self.mods}) == 1
        self.planner = AdaLoRAPlanner(len(self.mods), self.mods[0].init_r, target_r, total_step, tinit, tfinal,
                                      deltaT, beta1, beta2)

    @staticmethod
    def _np(t):
        return t.detach().float().cpu().numpy()

    def step(self, global_step):
        pl = self.planner
        if global_step < pl.total_step - pl.tfinal:
            grads = [g for m in self.mods for g in (m.A.grad, m.E.grad, m.B.grad)]
            if all(g is not None for g in grads) and bool(torch.stack([torch.isfinite(g).all() for g in grads]).all()):
                pl.update_importance(
                    [{"A": self._np(m.A), "E": self._np(m.E), "B": self._np(m.B)} for m in self.mods],
                    [{"A": self._np(m.A.grad), "E": self._np(m.E.grad), "B": self._np(m.B.grad)} for m in self.mods])
        keep = pl.plan(global_step)
        if keep is not None:
            with torch.no_grad():
                for m, k in zip(self.mods, keep):
                    m.E.mul_(torch.as_tensor(k, device=m.E.device, dtype=m.E.dtype))   


@torch.no_grad()
def adalora_rank_report(model):
    rows = [(n, int((m.E != 0).sum().item()), m.init_r) for n, m in model.named_modules() if isinstance(m, SVDLinear)]
    for n, kept, init in rows:
        print(f"{n:45s} rank {kept:2d} / {init}")
    print(f"average rank {np.mean([r[1] for r in rows]):.2f} over {len(rows)} matrices (total {sum(r[1] for r in rows)})")
    return rows

def freeze_for_adapters(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, (LoRMALinear, SVDLinear)):
            for n in ("A", "B", "E"):
                if hasattr(m, n):
                    getattr(m, n).requires_grad = True
    for p in model.head_parameters():
        p.requires_grad = True


@torch.no_grad()
def merge_adapters(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, (LoRMALinear, SVDLinear)):
                setattr(m, cn, c.to_linear())


def build_lsi_model(build_fn, ckpt, head_path, D, pool="last", agg="mean", dropout=0.1):
    backbone = load_phase1(build_fn, ckpt)
    prior = D["pos_counts"] / max(1, len(D["train"]))          
    return LSIModel(backbone, D["num_labels"], head_path=head_path, pool=pool, agg=agg, dropout=dropout, prior=prior)


def make_val_fn(D, DOC, device, batch_size=2):
    Y = docs_to_Y(D["dev"], D["num_labels"])

    def val_fn(m):
        probs = predict_lsi(m, D["dev"], DOC, D["pad_id"], D["sep_id"], device, batch_size)
        return tune_global_threshold(probs, Y, min_labels=1)[1]["macro_f1"]
    return val_fn

def run_lorma_lsi(build_fn, ckpt, head_path, D, DOC, device, tag="lsi_lorma", variant="plus", r=8, alpha=16,targets=("q_proj", "v_proj"), lora_dropout=0.0, pool="last", agg="mean", cfg=None):
    cfg = cfg or CFG_LSI
    model = build_lsi_model(build_fn, ckpt, head_path, D, pool, agg)
    inject_lorma(model.backbone, targets, variant, r, alpha, lora_dropout)
    freeze_for_adapters(model)
    count_params(model)
    pw = make_pos_weight(D["pos_counts"], len(D["train"]), cfg.get("pos_weight"))
    hist = train_lsi(model, D["train"], D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=make_val_fn(D, DOC, device),save_path=f"/kaggle/working/{tag}_best.pt", pos_weight=pw)
    return model, hist


def run_adalora_lsi(build_fn, ckpt, head_path, D, DOC, device, tag="lsi_adalora", targets=("q_proj", "v_proj"),
                    init_r=12, target_r=8, alpha=16, dropout=0.1, tinit=0, tfinal=0, deltaT=1, beta1=0.85, beta2=0.85,
                    orth_reg_weight=0.5, pool="last", agg="mean", cfg=None):
    cfg = cfg or CFG_LSI
    model = build_lsi_model(build_fn, ckpt, head_path, D, pool, agg)
    inject_adalora(model.backbone, targets, init_r, alpha, dropout)
    freeze_for_adapters(model)
    count_params(model)
    total_step = lsi_total_steps(D["train"], cfg)
    print("optimizer steps (AdaLoRA total_step):", total_step)
    alloc = AdaLoRAAllocator(model.backbone, total_step, target_r, tinit, tfinal, deltaT, beta1, beta2)
    pw = make_pos_weight(D["pos_counts"], len(D["train"]), cfg.get("pos_weight"))
    hist = train_lsi(model, D["train"], D["pad_id"], D["sep_id"], DOC, cfg, device, val_fn=make_val_fn(D, DOC, device),save_path=f"/kaggle/working/{tag}_best.pt", pos_weight=pw,extra_loss=lambda m: orth_reg_weight * adalora_orth_reg(m),on_step=lambda m, step: alloc.step(step))
    adalora_rank_report(model)
    return model, hist


def check_adapter_identity_and_merge(build_fn, ckpt, kind="adalora", targets=("q_proj", "v_proj"), **kw):
    ids = torch.randint(0, 1000, (1, 64))
    plain = load_phase1(build_fn, ckpt).eval()
    wrapped = load_phase1(build_fn, ckpt)
    (inject_adalora if kind == "adalora" else inject_lorma)(wrapped, targets, **kw)
    wrapped.eval()
    with torch.no_grad():
        d0 = (plain(ids)["logits"] - wrapped(ids)["logits"]).abs().max().item()
        for m in wrapped.modules():
            if isinstance(m, (LoRMALinear, SVDLinear)):
                for n in ("A", "B", "E"):
                    if hasattr(m, n):
                        nn.init.normal_(getattr(m, n), std=0.02)
        before = wrapped(ids)["logits"]
        merge_adapters(wrapped)
        d1 = (before - wrapped(ids)["logits"]).abs().max().item()
    print(f"identity at init: max|diff| = {d0:.2e} | merged vs factored: max|diff| = {d1:.2e} ")


def check_adapter_grads(model, device, alloc=None):
    model.to(device).train()
    out = model(*dummy_lsi_batch(model.vocab_size, device))
    out.float().sum().backward()
    bad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
    print("trainable params without a gradient:", bad[:5] if bad else "none")
    if alloc is not None:
        alloc.step(1)
        print("allocator step OK")

In [ ]:
def student_by_truncation(build_fn, model_pt, head_path, D, keep_layers=5, pool="last", agg="mean", blocks_attr="blocks"):
    bb = load_phase1(build_fn, model_pt)
    blocks = getattr(bb, blocks_attr)
    n = len(blocks)
    idx = sorted(set(int(round(i)) for i in np.linspace(0, n - 1, keep_layers)))
    setattr(bb, blocks_attr, nn.ModuleList([blocks[i] for i in idx]))
    print(f"student keeps blocks {idx} of {n}")
    prior = (D["pos_counts"] + 0.5) / (len(D["train"]) + 1)
    return LSIModel(bb, D["num_labels"], head_path=head_path, pool=pool, agg=agg, prior=prior)


def student_from_model(student_build_fn, student_pt, head_path, D, pool="last", agg="mean"):
    prior = (D["pos_counts"] + 0.5) / (len(D["train"]) + 1)
    return LSIModel(load_phase1(student_build_fn, student_pt), D["num_labels"], head_path=head_path, pool=pool, agg=agg, prior=prior)


def rebuild_teacher(build_fn, model_pt, head_path, D, ckpt_path, device, adapt=None, pool="last", agg="mean"):
    model = LSIModel(load_phase1(build_fn, model_pt), D["num_labels"], head_path=head_path, pool=pool, agg=agg)
    if adapt is None:
        for p in model.parameters():
            p.requires_grad = True
    else:
        model = adapt(model)
    sd = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(sd, strict=False)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    print(f"loaded {len(sd)} tensors; trainable params missing from the checkpoint: {len(trainable - set(sd))}; "
          f"checkpoint tensors with no matching parameter: {len(set(sd) - {n for n, _ in model.named_parameters()})}   (both should be 0)")
    return model.to(device).eval()

@torch.no_grad()
def logits_lsi(model, docs, DOC, pad_id, sep_id, device, batch_size=2, use_amp=True, desc="teacher"):
    model.eval()
    amp = use_amp and device.type == "cuda"
    L = model.num_labels
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["ids"]))
    out = np.zeros((len(docs), L), dtype=np.float32)
    for s in tqdm(range(0, len(order), batch_size), desc=desc, leave=False):
        idxs = order[s:s + batch_size]
        b = collate_lsi([docs[i] for i in idxs], pad_id, DOC, sep_id, L, training=False)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            lg = model(b["input_ids"].to(device), b["lengths"].to(device), b["doc_idx"].to(device),
                       b["chunk_pos"].to(device), b["n_docs"])
        out[idxs] = lg.float().cpu().numpy()
    return out


def teacher_logits(teachers, D, device, split="train", batch_size=2):
    acc = 0
    for t in teachers:
        model, DOC, *rest = t
        Dt = rest[0] if rest else D
        acc = acc + logits_lsi(model, Dt[split], DOC, Dt["pad_id"], Dt["sep_id"], device, batch_size)
    return acc / len(teachers)


def attach_soft(docs, soft, teacher_ids=None):
    return [{**d, "soft": soft[i]} for i, d in enumerate(docs)]

def collate_kd(batch, pad_id, DOC, sep_id, num_labels, training=False, rng=None):
    b = collate_lsi(batch, pad_id, DOC, sep_id, num_labels, training, rng)
    b["soft"] = torch.from_numpy(np.stack([it["soft"] for it in batch]).astype(np.float32))
    return b


def kd_loss_ml(logits, Y, soft_logits, T, kd_alpha, loss="bce", pos_weight=None, gamma=2.0):
    hard = lsi_loss(logits, Y, loss, pos_weight, gamma)
    zs, zt = logits.float() / T, soft_logits / T
    p = torch.sigmoid(zt)
    kl = p * (F.logsigmoid(zt) - F.logsigmoid(zs)) + (1 - p) * (F.logsigmoid(-zt) - F.logsigmoid(-zs))
    kd = kl.mean() * (T * T)
    return kd_alpha * hard + (1 - kd_alpha) * kd, hard.detach(), kd.detach()


def train_lsi_kd(model, docs, pad_id, sep_id, DOC, cfg, device, val_fn=None, save_path=None, pos_weight=None,T=2.0, kd_alpha=0.5):
    model.to(device)
    rng = np.random.default_rng(cfg.get("seed", 0))
    L = model.num_labels
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

    loader = DataLoader(docs, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_kd(b, pad_id, DOC, sep_id, L, training=True, rng=rng))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    pw = pos_weight.to(device) if pos_weight is not None else None
    best, history = -1.0, []
    for epoch in range(cfg["epochs"]):
        model.train()
        run, run_h, run_k, nb, t0 = 0.0, 0.0, 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            ids, ln, di, cp = (b[k].to(device) for k in ("input_ids", "lengths", "doc_idx", "chunk_pos"))
            Y, soft = b["labels"].to(device), b["soft"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(ids, ln, di, cp, b["n_docs"])
            loss, hard, kd = kd_loss_ml(logits, Y, soft, T, kd_alpha, cfg.get("loss", "bce"), pw, cfg.get("focal_gamma", 2.0))
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); run_h += hard.item(); run_k += kd.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        nb = max(1, nb)
        rec = {"epoch": epoch + 1, "train_loss": run / nb, "hard": run_h / nb, "kd": run_k / nb, "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history

CFG_KD = {**CFG_LSI, "lr_backbone": 5e-5, "lr_head": 1e-3}          


def run_kd_lsi(teachers, student, D, DOC, device, tag="lsi_kd", T=2.0, kd_alpha=0.5, D_student=None,cfg=None, train_subset=None, eval_batch_size=2):
    cfg = cfg or CFG_KD
    Ds = D_student or D
    L = D["num_labels"]
    ctx = getattr(student.backbone, "context_length", None)
    assert ctx is None or ctx >= DOC["max_len"], "student context length < DOC['max_len']"
    soft = teacher_logits(teachers, D, device, "train")
    Y_train = docs_to_Y(D["train"], L)
    tp = decode_probs(1 / (1 + np.exp(-soft)), 0.5, min_labels=1)
    s_t = multilabel_scores(Y_train, tp)
    probs_t = 1 / (1 + np.exp(-soft))
    print(f"teacher on the training documents (thr 0.5): macro-F1 {s_t['macro_f1']:.3f}, micro-F1 {s_t['micro_f1']:.3f}; "
          f"mean probability on gold statutes {probs_t[Y_train == 1].mean():.3f}, on the others {probs_t[Y_train == 0].mean():.4f}"
          f"  (near 1 / near 0 = very sharp targets: raise T)")
    docs = attach_soft(Ds["train"], soft, [d["id"] for d in D["train"]])
    if train_subset:
        pick = np.random.default_rng(cfg.get("seed", 0)).choice(len(docs), size=min(train_subset, len(docs)), replace=False)
        docs = [docs[i] for i in sorted(pick)]
    for p in student.parameters():
        p.requires_grad = True
    count_params(student)
    print(f"KD on {len(docs)} documents, {L} statutes, T={T}, kd_alpha={kd_alpha}; student document representation: {DOC}")

    Y_dev = docs_to_Y(Ds["dev"], L)

    def val_fn(m):
        probs = predict_lsi(m, Ds["dev"], DOC, Ds["pad_id"], Ds["sep_id"], device, eval_batch_size)
        t, s = tune_global_threshold(probs, Y_dev, min_labels=1)
        print_scores(s, f"   dev (thr {t:.2f}):")
        return s["macro_f1"]

    counts = label_counts(docs, L)
    pw = make_pos_weight(counts, len(docs), cfg.get("pos_weight"))
    hist = train_lsi_kd(student, docs, Ds["pad_id"], Ds["sep_id"], DOC, cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/{tag}_best.pt", pos_weight=pw, T=T, kd_alpha=kd_alpha)
    return student, hist

def kd_report(models, device, n_docs=None):
    for name, (m, DOC, Dm) in models.items():
        docs = Dm["dev"][:n_docs] if n_docs else Dm["dev"]
        n_par = sum(p.numel() for p in m.parameters())
        mb = sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        probs = predict_lsi(m, docs, DOC, Dm["pad_id"], Dm["sep_id"], device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        t, s = tune_global_threshold(probs, docs_to_Y(docs, Dm["num_labels"]), min_labels=1)
        print(f"{name:10s} params {n_par/1e6:7.2f}M ({mb:6.0f} MB) | dev macro-F1 {s['macro_f1']:.4f}, micro-F1 {s['micro_f1']:.4f},"f"subset acc {s['subset_acc']:.4f} @thr {t:.2f} | dev inference {sec:.0f}s")

In [ ]:
P5 = "Phase_5_LSI"
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
p5 = download_lsi_files(hf_dataset_repo, P5, HF_TOKEN)                       
D = setup_data(p5, "/kaggle/working/legal_bpe_tokenizer.pkl", cache="/kaggle/working/lsi_tok.pkl")
PAD_ID, VOCAB_SIZE = D["pad_id"], D["vocab_size"]

In [ ]:
model_full, hist_full = run_full_ft_lsi(build_model, MODEL_PT, HEAD_PATH, D, DOC_CHUNKS, device, tag="lsi_full",agg="logit_max", cfg={**CFG_LSI, "pos_weight": "sqrt"})
report = lsi_report(model_full, D, DOC_CHUNKS, device)

In [ ]:
DOC_TEACHER = DOC_CHUNKS            
DOC_STUDENT = DOC_TAIL             

student = student_by_truncation(build_model, MODEL_PT, HEAD_PATH, D, keep_layers=5)
student, hist_kd = run_kd_lsi([(model_full, DOC_TEACHER)], student, D, DOC_STUDENT, device, tag="lsi_kd")
kd_report({"teacher": (model_full, DOC_TEACHER, D), "student": (student, DOC_STUDENT, D)}, device)

In [ ]:
p5 = download_lsi_files(hf_dataset_repo, P5, HF_TOKEN, include_test=True)
raw_test = json.load(open(p5["test"], encoding="utf-8"))
preds = predict_cases(student, raw_test, D, DOC_CHUNKS, device, report["global_t"])
write_lsi_submission(preds, valid_ids=D["labels"], zip_name="submission_lsi_kd.zip")